# Crear y Compilar la Red TinyML 1D-CNN + TCN para MyoTensor (ESP32-S3)

Este cuaderno define la arquitectura temporal **TinyML 1D-CNN + TCN (Temporal Convolutional Network)** optimizada para inferencia en tiempo real en microcontrolador.
- **Entrada:** Ventana de 200 ms ($F_s = 1000$ Hz, tensor $(200, 1)$).
- **Downsampling Temprano:** Convolución de entrada con `strides=2` ($L = 200 \to 100$).
- **Convoluciones Causales Dilatadas:** Bloques residuales con dilataciones $d \in \{1, 2, 4\}$ y **16 filtros** alineados a registros vectoriales de 128 bits.
- **100% Compatible con SIMD Xtensa LX7:** Instrucciones vectoriales `ESP-NN` de 1 ciclo (`EE.VMUL.S8.ACC`).
- **Salida:** `GlobalAveragePooling1D -> Dense(16) -> Dense(4, softmax)`.

In [1]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import tensorflow as tf
from pathlib import Path

# Carga de variables de entorno desde .env
def load_env_variables():
    try:
        start_dir = Path(os.getcwd())
    except:
        start_dir = Path(".")
        
    env_path = None
    for path in [start_dir] + list(start_dir.parents):
        temp_path = path / ".env"
        if temp_path.exists():
            env_path = temp_path
            break
    if env_path is None:
        raise FileNotFoundError("⚠️ No se pudo encontrar el archivo .env en la raíz del proyecto.")
    with open(env_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, val = line.split("=", 1)
            os.environ[key.strip()] = val.strip()
    print(f"✅ Archivo .env cargado con éxito desde: {env_path}")

load_env_variables()
models_dir = os.environ["MODELS_DL_PROTO"]
os.makedirs(models_dir, exist_ok=True)

I0000 00:00:1787777633.543379   46806 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1787777633.612076   46806 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1787777635.560352   46806 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


✅ Archivo .env cargado con éxito desde: /home/cbe/Proyectos/MyoTensor_Tesis/.env


In [2]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv1D, Add, Activation, GlobalAveragePooling1D, Dense
)

def tcn_residual_block(x, filters=16, kernel_size=3, dilation_rate=1, block_id=0):
    """
    Construye un bloque residual TCN optimizado para SIMD de 128 bits (16 filtros).
    """
    prev_x = x
    
    conv1 = Conv1D(
        filters=filters,
        kernel_size=kernel_size,
        dilation_rate=dilation_rate,
        padding="causal",
        activation="relu",
        name=f"tcn_conv1d_a_{block_id}_d{dilation_rate}"
    )(x)
    
    conv2 = Conv1D(
        filters=filters,
        kernel_size=kernel_size,
        dilation_rate=dilation_rate,
        padding="causal",
        activation="relu",
        name=f"tcn_conv1d_b_{block_id}_d{dilation_rate}"
    )(conv1)
    
    out = Add(name=f"tcn_add_{block_id}")([prev_x, conv2])
    return Activation("relu", name=f"tcn_out_act_{block_id}")(out)

def build_simd_tcn_model(
    window_size=200, 
    num_channels=1, 
    num_classes=4, 
    nb_filters=16, 
    kernel_size=3, 
    dilations=(1, 2, 4), 
    learning_rate=1e-3
):
    """
    Construye y compila el modelo TinyML-TCN optimizado para ESP32-S3 SIMD.
    """
    inputs = Input(shape=(window_size, num_channels), name="input_semg")
    
    # 1. Convolución de entrada con downsample 2x (L=200 -> L=100)
    x = Conv1D(
        filters=nb_filters, 
        kernel_size=kernel_size, 
        strides=2, 
        padding="same", 
        activation="relu", 
        name="entry_conv"
    )(inputs)
    
    # 2. Pila de Bloques Residuales TCN con Dilatación Creciente
    for i, d in enumerate(dilations):
        x = tcn_residual_block(
            x=x, 
            filters=nb_filters, 
            kernel_size=kernel_size, 
            dilation_rate=d, 
            block_id=i+1
        )
        
    # 3. Clasificador Vectorizado
    x = GlobalAveragePooling1D(name="tcn_gap")(x)
    x = Dense(16, activation="relu", name="dense_features")(x)
    outputs = Dense(num_classes, activation="softmax", name="output_gestures")(x)
    
    model = Model(inputs=inputs, outputs=outputs, name="SIMD_TCN_MyoTensor")
    
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(
        optimizer=optimizer,
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

In [3]:
# Instanciar y revisar resumen de la arquitectura optimizada
model = build_simd_tcn_model(window_size=200, num_channels=1, num_classes=4, nb_filters=16)
model.summary()

# Guardar modelo base en formato .keras
model_save_path = os.path.join(models_dir, "myotensor_proto_net_tcn.keras")
print(f"Guardando modelo base en: {model_save_path} ...")
model.save(model_save_path)
print("✅ ¡Modelo TinyML SIMD-TCN base guardado con éxito!")

I0000 00:00:1787777638.464866   46806 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2279 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2050, pci bus id: 0000:01:00.0, compute capability: 8.6


Model: "SIMD_TCN_MyoTensor"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_semg (InputLayer)     [(None, 200, 1)]             0         []                            
                                                                                                  
 entry_conv (Conv1D)         (None, 100, 16)              64        ['input_semg[0][0]']          
                                                                                                  
 tcn_conv1d_a_1_d1 (Conv1D)  (None, 100, 16)              784       ['entry_conv[0][0]']          
                                                                                                  
 tcn_conv1d_b_1_d1 (Conv1D)  (None, 100, 16)              784       ['tcn_conv1d_a_1_d1[0][0]']   
                                                                                 